In [1]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
def read_parquet_by_type(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
    purchase_history_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
    item_chunk_files = [file for file in files if 'item_chunk' in file]
    
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
    purchase_history_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_history_chunk_files]) if purchase_history_chunk_files else None
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
    
    # Trả về một dictionary chứa các DataFrame
    return {
        "user_chunk": user_chunk_df,
        "purchase_history_chunk": purchase_history_chunk_df,
        "item_chunk": item_chunk_df
    }

In [3]:
train_path = 'E:/KHMT2023_CS_UIT/05_C_Python_For_ML/recommendation_dataset'
dataframes = read_parquet_by_type(train_path)

df_user = dataframes["user_chunk"]
df_purchase = dataframes["purchase_history_chunk"]
df_item = dataframes["item_chunk"]

In [4]:
print("User columns:", df_user.columns)
print("Purchase columns:", df_purchase.columns)
print("Item columns:", df_item.columns)


User columns: ['customer_id', 'gender', 'location', 'province', 'membership', 'timestamp', 'created_date', 'updated_date', 'sync_status_id', 'last_sync_date', 'sync_error_message', 'region', 'location_name', 'install_app', 'install_date', 'district', 'user_id', 'is_deleted']
Purchase columns: ['timestamp', 'user_id', 'item_id', 'event_type', 'event_value', 'price', 'date_key', 'quantity', 'customer_id', 'created_date', 'updated_date', 'channel', 'payment', 'location', 'discount', 'is_deleted']
Item columns: ['p_id', 'item_id', 'price', 'category_l1_id', 'category_l1', 'category_l2_id', 'category_l2', 'category_l3_id', 'category_l3', 'category_id', 'category', 'description', 'brand', 'manufacturer', 'creation_timestamp', 'is_deleted', 'created_date', 'updated_date', 'sync_status_id', 'last_sync_date', 'sync_error_message', 'image_url', 'gender_target', 'age_group', 'item_type', 'gp', 'weight', 'color', 'size', 'origin', 'volume', 'material', 'sale_status', 'description_new']


## Preprocessing User


In [5]:
import polars as pl

def user_preprocessing(df_user: pl.DataFrame) -> pl.DataFrame:
    """
    Tiền xử lý dữ liệu user:
    - Chuẩn hóa province (xóa 'Thành Phố', 'Tỉnh')
    - Chuẩn hóa gender (Khác -> Nữ)
    - Chuẩn hóa install_app (mapping theo quy tắc)

    Trả về df_user đã được xử lý.
    """

    keep_col_user = [
    "customer_id",
    "gender",
    "province",
    "membership",
    "timestamp",
    "created_date",
    "region",
    "install_app",
    "user_id"
    ]

    df_user = df_user.select(keep_col_user)

    # 1️⃣ Chuẩn hóa 'province'
    df_user = df_user.with_columns(
        pl.col('province')
            .str.replace('Thành Phố ', '')
            .str.replace('Tỉnh ', '')
            .alias('province')
    )

    # 2️⃣ Chuẩn hóa gender ('Khác' -> 'Nữ')
    df_user = df_user.with_columns(
        pl.col("gender").replace({"Khác": "Nữ"}).alias("gender")
    )

    # 3️⃣ Chuẩn hóa install_app theo mapping
    mapping = {
        "In-Store": "In-Store",
        "iOS": "Mobile",
        "Android": "Mobile",
        "SPE": "SPE",
        "Call": "Other",
        "CRM Partner": "Other",
        "Wholesale": "Other",
        "Chat": "Other",
        "Không xác định": "Other",
        "LZD": "Other",
        "Web": "Other"
    }

    df_user = df_user.with_columns(
        pl.col("install_app").replace(mapping).alias("install_app")
    )

    return df_user


In [6]:
df_user = user_preprocessing(df_user)

In [7]:
print(df_user.head())

shape: (5, 9)
┌────────────┬────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ customer_i ┆ gender ┆ province   ┆ membershi ┆ … ┆ created_d ┆ region    ┆ install_a ┆ user_id   │
│ d          ┆ ---    ┆ ---        ┆ p         ┆   ┆ ate       ┆ ---       ┆ pp        ┆ ---       │
│ ---        ┆ str    ┆ str        ┆ ---       ┆   ┆ ---       ┆ str       ┆ ---       ┆ str       │
│ i32        ┆        ┆            ┆ str       ┆   ┆ datetime[ ┆           ┆ str       ┆           │
│            ┆        ┆            ┆           ┆   ┆ μs]       ┆           ┆           ┆           │
╞════════════╪════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 14732      ┆ Nam    ┆ Hồ Chí     ┆ Standard  ┆ … ┆ 2011-05-2 ┆ Đông Nam  ┆ In-Store  ┆ e1e482066 │
│            ┆        ┆ Minh       ┆           ┆   ┆ 5 21:11:5 ┆ Bộ        ┆           ┆ 52bf8c279 │
│            ┆        ┆            ┆           ┆   ┆ 1.677     ┆           ┆ 

## Preprocessing Item

In [9]:
import polars as pl

def item_preprocessing(df_item: pl.DataFrame) -> pl.DataFrame:
    """
    Tiền xử lý bảng item:
    - Giữ lại các cột quan trọng theo keep_col_item
    - Tạo cột price_log từ price bằng log1p
    """

    # 1️⃣ Giữ lại danh sách cột cần thiết
    keep_col_item = [
        "p_id",
        "item_id",
        "price",
        "category_l1_id",
        "category_l1",
        "category_l2_id",
        "category_l2",
        "category_l3_id",
        "category_l3",
        "category_id",
        "category",
        "description",
        "brand",
        "manufacturer",
        "created_date",
        "gender_target",
        "age_group",
        "item_type",
        "gp",
        "color",
        "size",
        "origin",
        "material",
        "description_new"
    ]

    # Select only keep columns (ignore missing cols safely)
    df_item = df_item.select([col for col in keep_col_item if col in df_item.columns])

    # 2️⃣ Tạo cột price_log
    if "price" in df_item.columns:
        df_item = df_item.with_columns(
            pl.col("price").log1p().alias("price_log")
        )

    # 3️⃣ (Optional) Chuẩn hoá text — hook mở rộng sau
    # df_item = df_item.with_columns(
    #     pl.col("brand").str.to_lowercase().alias("brand")
    # )

    return df_item



In [10]:
df_item = item_preprocessing(df_item)
print(df_item.head())

shape: (5, 25)
┌───────┬────────────┬────────────┬────────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ p_id  ┆ item_id    ┆ price      ┆ category_l ┆ … ┆ origin    ┆ material  ┆ descripti ┆ price_log │
│ ---   ┆ ---        ┆ ---        ┆ 1_id       ┆   ┆ ---       ┆ ---       ┆ on_new    ┆ ---       │
│ i32   ┆ str        ┆ decimal[38 ┆ ---        ┆   ┆ str       ┆ str       ┆ ---       ┆ f64       │
│       ┆            ┆ ,4]        ┆ i32        ┆   ┆           ┆           ┆ str       ┆           │
╞═══════╪════════════╪════════════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 17065 ┆ 0502020000 ┆ 99000.0000 ┆ 1          ┆ … ┆ Không xác ┆ Không xác ┆ Chi tiết  ┆ 11.502885 │
│       ┆ 004        ┆            ┆            ┆   ┆ định      ┆ định      ┆ sản phẩm  ┆           │
│       ┆            ┆            ┆            ┆   ┆           ┆           ┆ …         ┆           │
│ 72370 ┆ 0010290040 ┆ 69000.0000 ┆ 3292       ┆ … ┆ Không xác ┆ Không xác ┆

## Preprocessing Purchase

In [11]:
import polars as pl

def purchase_preprocessing(df_purchase: pl.DataFrame) -> pl.DataFrame:
    """
    Tiền xử lý bảng purchase:
    - Giữ lại các cột quan trọng theo keep_col_purchase
    - Chuẩn hoá kênh mua hàng (channel) theo mapping
    """

    # 1️⃣ Giữ lại danh sách cột quan trọng
    keep_col_purchase = [
        "timestamp",
        "user_id",
        "item_id",
        "price",
        "date_key",
        "quantity",
        "customer_id",
        "created_date",
        "updated_date",
        "channel",
        "payment",
        "location",
        "discount"
    ]

    # Select chỉ những cột tồn tại trong dataframe
    df_purchase = df_purchase.select([c for c in keep_col_purchase if c in df_purchase.columns])

    # 2️⃣ Chuẩn hoá channel bằng mapping
    mapping = {
        "In-Store": "In-Store",
        "iOS": "Mobile",
        "Android": "Mobile",
        "SPE": "SPE",
        "Call": "Other",
        "CRM Partner": "Other",
        "Wholesale": "Other",
        "Chat": "Other",
        "Không xác định": "Other",
        "LZD": "Other",
        "TKS": "Other",
        "Web": "Other"
    }

    df_purchase = df_purchase.with_columns(
        pl.col("channel").replace(mapping).alias("channel")
    )

    # 3️⃣ (Optional – mở rộng sau)
    # Chuyển timestamp về dạng datetime
    # if "timestamp" in df_purchase.columns:
    #     df_purchase = df_purchase.with_columns(
    #         pl.col("timestamp").str.strptime(pl.Datetime, fmt="%Y-%m-%d %H:%M:%S", strict=False)
    #     )

    # Xử lý price / quantity nếu cần
    # df_purchase = df_purchase.with_columns(
    #     (pl.col("price") * pl.col("quantity")).alias("total_spent")
    # )

    return df_purchase



In [12]:
df_purchase = purchase_preprocessing(df_purchase)
print(df_purchase.head())


shape: (5, 13)
┌────────────┬────────────┬───────────┬───────────┬───┬──────────┬──────────┬──────────┬───────────┐
│ timestamp  ┆ user_id    ┆ item_id   ┆ price     ┆ … ┆ channel  ┆ payment  ┆ location ┆ discount  │
│ ---        ┆ ---        ┆ ---       ┆ ---       ┆   ┆ ---      ┆ ---      ┆ ---      ┆ ---       │
│ i64        ┆ str        ┆ str       ┆ decimal[3 ┆   ┆ str      ┆ str      ┆ i32      ┆ decimal[3 │
│            ┆            ┆           ┆ 8,4]      ┆   ┆          ┆          ┆          ┆ 8,4]      │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪══════════╪══════════╪══════════╪═══════════╡
│ 1735064221 ┆ ca12702ddf ┆ 711500000 ┆ 49000.000 ┆ … ┆ In-Store ┆ VietQR   ┆ 656      ┆ 0.0000    │
│            ┆ 55acaa9fb7 ┆ 0004      ┆ 0         ┆   ┆          ┆          ┆          ┆           │
│            ┆ 67e10faaa6 ┆           ┆           ┆   ┆          ┆          ┆          ┆           │
│            ┆ …          ┆           ┆           ┆   ┆          ┆          

In [37]:
df_merge = df_purchase.select(["item_id", "price", "quantity", "discount"]) \
    .join(
        df_item.select(["item_id", pl.col("price").alias("item_price")]),
        on="item_id",
        how="left"
    )


In [38]:
print(df_merge)

shape: (35_729_825, 5)
┌───────────────┬───────────────┬──────────┬───────────────┬───────────────┐
│ item_id       ┆ price         ┆ quantity ┆ discount      ┆ item_price    │
│ ---           ┆ ---           ┆ ---      ┆ ---           ┆ ---           │
│ str           ┆ decimal[38,4] ┆ i32      ┆ decimal[38,4] ┆ decimal[38,4] │
╞═══════════════╪═══════════════╪══════════╪═══════════════╪═══════════════╡
│ 7115000000004 ┆ 49000.0000    ┆ 1        ┆ 0.0000        ┆ 49000.0000    │
│ 0029130000030 ┆ 69000.0000    ┆ 1        ┆ 0.0000        ┆ 74000.0000    │
│ 3496000000053 ┆ 75000.0000    ┆ 2        ┆ 0.0000        ┆ 75000.0000    │
│ 2700000000002 ┆ 58500.0000    ┆ 2        ┆ 13000.0000    ┆ 65000.0000    │
│ 0029110000036 ┆ 89000.0000    ┆ 1        ┆ 10000.0000    ┆ 99000.0000    │
│ …             ┆ …             ┆ …        ┆ …             ┆ …             │
│ 1396000000020 ┆ 28050.0000    ┆ 1        ┆ 4950.0000     ┆ 36000.0000    │
│ 0007070000403 ┆ 189000.0000   ┆ 1        ┆ 0.0000  